<div class="alert alert-block alert-info" style="padding:20px;color:white;margin:auto;font-size:300%;text-align:center;display:fill;border-radius:60px;background-color:#37006f;overflow:hidden;font-weight:800">Midi AutoRegressive Generation</div>

## 1. Setup

In [1]:
cd ..

d:\Personal Projects\tune-ml


d:\Personal Projects\tune-ml\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# imports
from tuneml.data.automidi import AutoMidiDataset

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("Midi GPT Training")

# Enable system metrics logging
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)  # Log system metrics every 1 second

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data Processing

In [5]:
datapath = "./datasets/midicaps/train.pt"
dataset = AutoMidiDataset(
    "datasets/midicaps/lmd_full",
    save_path=datapath,
    n_samples=10000,
)
print(f"Dataset size: {len(dataset)}")
print(f"Example MIDI shape: {dataset.midi[0].shape}")

d:\Personal Projects\tune-ml\tuneml\tokenizers\MidiTokenizer.py:57: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  self.tokenizer = Structured(config)


MIDI shape: torch.Size([10000, 2048])
Dataset size: 10000
Example MIDI shape: torch.Size([2048])


## 3. Model Training

In [3]:
# Hyperparameters
datapath = "./datasets/midicaps/train.pt"
batch_size = 1
warmup_steps = 2000
ckpt_path = "./weights/midi_gpt_ckpt.pt"
load_checkpoint = False
save_path = "./weights/midi_gpt.pt"
num_epochs = 30
hparams = {
    "d_model": 256,
    "num_layers": 4,
    "num_heads": 8,
    "d_ff": 512,
    "max_midi_len": 2048,
    "bias": True,
    "dropout": 0.1,
    "layernorm_eps": 1e-6,
}

In [ ]:
import torch
from datetime import datetime
from tuneml.trainers.MidiGPTTrainer import MidiGPTTrainer

# set up the trainer
print("Setting up the trainer...")
trainer = MidiGPTTrainer(
    hparams,
    datapath,
    batch_size,
    warmup_steps,
    ckpt_path,
    load_checkpoint,
    mlflow_enabled=True,
    mlflow_log_model=True
 )
print()

# train the model
run_name = "model_training_" + datetime.now().strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=run_name):
    try:
        trainer.fit(num_epochs)
    except Exception as e:
        print(f"Error occurred during training: {e}")

# done training, save the model
print("Saving...")
save_file = {
    "model_state_dict": trainer.model.state_dict(),
    "hparams": trainer.hparams
}
torch.save(save_file, save_path)
print("Done!")

Setting up the trainer...
There are 10000 samples in the data, 8000 training samples and 2000 validation samples



d:\Personal Projects\tune-ml\tuneml\tokenizers\MidiTokenizer.py:57: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  self.tokenizer = Structured(config)
2026/05/21 20:30:18 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/21 20:30:18 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Beginning training...
2026-05-21 20:30


Training Epoch: 1/30:   0%|          | 0/30 [00:00<?, ?it/s]d:\Personal Projects\tune-ml\tuneml\trainers\MidiGPTTrainer.py:73: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  midi_seqs = [torch.tensor(seq, dtype=torch.long) for seq in midi_seqs]
d:\Personal Projects\tune-ml\tuneml\trainers\MidiGPTTrainer.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  return float(loss)
Training Epoch: 1/30:   0%|          | 0/30 [1:36:46<?, ?it/s]
2026/05/21 22:07:04 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


Checkpointing...
Done
2026-05-21 22:07
🏃 View run model_training_20260521_203017 at: http://127.0.0.1:8080/#/experiments/3/runs/6c03b09bcd6c44bcb780d1d838565025
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


In [ ]:
import matplotlib.pyplot as plt

train_losses = list(getattr(trainer, "train_losses", []))
val_losses = list(getattr(trainer, "val_losses", []))

if not train_losses or not val_losses:
    raise ValueError("No training history found. Run trainer.fit(...) before plotting.")

epochs = range(1, min(len(train_losses), len(val_losses)) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses[:len(epochs)], label="Train Loss", linewidth=2)
plt.plot(epochs, val_losses[:len(epochs)], label="Validation Loss", linewidth=2)
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Model Evaluation

In [2]:
from tuneml.modules.generator.MidiGenerator import MidiGenerator

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# set up the generator with the best model
model_ckpt_path = "./weights/midi_gpt.pt"
generator = MidiGenerator(model_ckpt_path=model_ckpt_path)

CUDA is not available, falling back to CPU


d:\Personal Projects\tune-ml\tuneml\tokenizers\MidiTokenizer.py:57: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  self.tokenizer = Structured(config)


In [4]:
# generate a happy MIDI from text
midi_tokens = generator(
    midi_file_path="D:/Personal Projects/tune-ml/datasets/jazz midi/2ndMovementOfSinisterFootwear.mid"
)

In [5]:
# convert the generated MIDI tokens back to a MIDI file
generator.tokens_to_midi(
    tokens=midi_tokens,
    store_path="./tests/midi_gpt_output.mid"
)